In [ ]:
from pyspark.sql import SparkSession
import os
import shutil

# --- Configuration ---
ICEBERG_VERSION = "1.5.0"
LOCAL_WAREHOUSE_PATH = "C:/data/data_files/iceberg/iceberg_warehouse"
CATALOG_NAME = "local"

# --- Stop existing SparkSession ---
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception:
    pass

# --- Clean warehouse ---
# if os.path.exists(LOCAL_WAREHOUSE_PATH):
#     print(f"Cleaning up old warehouse: {LOCAL_WAREHOUSE_PATH}")
#     shutil.rmtree(LOCAL_WAREHOUSE_PATH)

# --- Iceberg packages ---
ICEBERG_PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION},"
    f"org.apache.avro:avro:1.11.3"
)

# --- SparkSession ---
spark = SparkSession.builder \
    .appName("IcebergDescribeExample2") \
    .config("spark.jars.packages", ICEBERG_PACKAGES) \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "hadoop") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

print("Spark version:", spark.version)

In [ ]:
# spark.sql("SHOW CATALOGS;").show(50, truncate = False)
# spark.sql("SHOW databases in local;").show(50, truncate = False)
# spark.sql("SHOW tables in local.HumanResources;").show(50, truncate = False)
# spark.sql("SHOW tables in local.Person;").show(50, truncate = False)
# spark.sql("SHOW tables in local.Production;").show(50, truncate = False)
# spark.sql("SHOW tables in local.Purchasing;").show(50, truncate = False)
spark.sql("SHOW tables in local.Sales;").show(50, truncate = False)


In [ ]:
from pyspark.sql import functions as F

soh = spark.table("local.Sales.SalesOrderHeader").alias("soh")
c = spark.table("local.Sales.Customer").alias("c")
p = spark.table("local.Person.Person").alias("p")
sod = spark.table("local.Sales.SalesOrderDetail").alias("sod")
prd = spark.table("local.Production.Product").alias("prd")

In [ ]:
# soh.printSchema()
# c.printSchema()
p.printSchema()
# sod.printSchema()
# prd.printSchema()

In [ ]:
from pyspark.sql import SparkSession
# spark.conf.set("spark.sql.caseSensitive", "false")
# SparkSession.builder.config("spark.driver.memory", "4g")

dfperson = spark.sql("Select * from local.Person.Person")
dfperson.show(10)


result_df = spark.sql("""
Select
soh.SalesOrderID,
soh.OrderDate,
soh.DueDate,
soh.ShipDate,
soh.Status,
soh.OnlineOrderFlag,
soh.SalesOrderNumber,
soh.PurchaseOrderNumber,
soh.SubTotal,
soh.TaxAmt,
soh.Freight,
soh.TotalDue,
soh.Comment
FROM    local.Sales.SalesOrderHeader soh   
""")

result_df.show()

# Print the schema
result_df.printSchema()

# spark.stop()


In [ ]:
from pyspark.sql import functions as F

joined_df = soh.join(
    c,
    soh["CustomerID"] == c["CustomerID"],
    how="inner"
) \
.join(
    p, c["PersonID"] == p["BusinessEntityID"],
    how="inner"
) \
.join(
    sod, soh["SalesOrderID"] == sod["SalesOrderID"],
    how="inner"
) \
.join(
    prd, sod["ProductID"] == prd["ProductID"],
    how="inner"
) \
.select(
    F.col("soh.SalesOrderID"),
    F.col("soh.OrderDate"),
    F.col("soh.DueDate"),
    F.col("soh.ShipDate"),
    F.col("soh.Status"),
    F.col("soh.OnlineOrderFlag"),
    F.col("soh.SalesOrderNumber"),
    F.col("soh.PurchaseOrderNumber"),
    F.col("soh.SubTotal"),
    F.col("soh.TaxAmt"),
    F.col("soh.Freight"),
    F.col("soh.TotalDue"),
    F.col("soh.Comment"),
    F.col("c.CustomerID"),
    F.col("p.firstName"),
    F.col("p.lastName"),
    F.col("c.AccountNumber"),
    F.col("prd.name").alias("ProductName"),
    F.col("sod.OrderQty"),
    F.col("sod.UnitPrice"),
    F.col("sod.LineTotal"),
    F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy"))
)

joined_df.show(5, truncate=False)

# joined_df.show(5, truncate=False)
# mayFilter = joined_df.filter((F.year(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 5) & (F.col("AccountNumber") == "AW00029825"))

# mayFilter.show(5, truncate=False)


In [ ]:
mayFilter.show(5, truncate=False)

In [ ]:
spark.sql("DESCRIBE TABLE local.Person.Person").show()

In [ ]:
p = spark.table("local.Person.Person").alias("p")
p.select("PersonType","NameStyle","Title","FirstName","MiddleName","LastName").show(10)

psql = spark.sql("Select * From local.Person.Person where Title is not NULL").alias("psql")
psql.show()